In [4]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/TiendaBigData.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 129 registros.
+--------------------+-------------------+--------+--------------------+-----+
|                 _id|      fecha_captura|   grupo|       identificador|valor|
+--------------------+-------------------+--------+--------------------+-----+
|69df9842151377f8d...|2026-04-15 13:52:26|Vannessa|18.5" Laptop, 24G...|489.0|
|69df9842151377f8d...|2026-04-15 13:52:26|Vannessa|Ryzen 5 7430U 15....|489.0|
|69df9842151377f8d...|2026-04-15 13:52:26|Vannessa|Laptop 15.6 Inch ...|349.0|
|69df9842151377f8d...|2026-04-15 13:52:26|Vannessa|15.6 Inch Laptop,...|289.0|
|69df9842151377f8d...|2026-04-15 13:52:26|Vannessa|Lenovo IdeaPad Sl...|449.0|
+--------------------+-------------------+--------+--------------------+-----+
only showing top 5 rows



In [1]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

NameError: name 'df' is not defined